# Часть I. Бустинги и Ансамблевые модели от scikit-learn

Жунёв Андрей Александрович РИМ-150950

**Цель:** Продемонстрировать работу алгоритмов бустингов и ансамлевых моделей от scikit-learn на примере датасете предсказания оценки вина(решение задачи регрессии). 

**Набор данных:** [Wine Quality (Red Wine Dataset)](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009)
**Источник:** Kaggle / UCI Machine Learning Repository  
**Размер:** 1599 образцов × 12 признаков  
**Целевая переменная:** quality (оценка 3-8 баллов)

## Оглавление

1. [I. Выбор и подготовка набора данных с кратким EDA](#I.-Выбор-и-подготовка-набора-данных-с-кратким-EDA)
2. [II. Предварительная обработка данных](#II.-Предварительная-обработка-данных)
3. [III. Обучение сложных моделей](#III.-Обучение-сложных-моделей-(до-2-баллов))
   - [1. Метрики для сравнения моделей](#1.-Метрики-для-сравнения-моделей)
   - [2. Выбор моделей](#2.-Выбор-моделей)
   - [3. Обучение Random Forest (базовая модель)](#3.-Обучение-Random-Forest-(базовая-модель))
   - [4. Обучение GradientBoostingRegressor (базовая модель)](#4.-Обучение-GradientBoostingRegressor-(базовая-модель))
   - [5. Сравнение базовых моделей](#5.-Сравнение-базовых-моделей)
   - [6. Выводы о базовых моделях](#6.-Выводы-о-базовых-моделях)
4. [IV. Оптимизация выбранных моделей](#IV.-Оптимизация-выбранных-моделей-(до-3-баллов))
   - [1. Выбор моделей для оптимизации](#1.-Выбор-моделей-для-оптимизации)
   - [2. Оптимизация Random Forest](#2.-Оптимизация-Random-Forest)
   - [3. Оптимизация GradientBoostingRegressor](#3.-Оптимизация-GradientBoostingRegressor)
5. [V. Итоговое сравнение оптимизированных моделей](#V.-Итоговое-сравнение-оптимизированных-моделей)
6. [VI. Промежуточные выводы](#VI.-Промежуточные-выводы)


## Импорты

In [140]:
# ============================================================================
# ИМПОРТЫ НЕОБХОДИМЫХ БИБЛИОТЕК
# ============================================================================

# Базовые библиотеки для работы с данными
import pandas as pd
import numpy as np
from pathlib import Path

# Визуализация (Plotly для интерактивных графиков)
import plotly.graph_objects as go  # Для детального контроля над графиками
import plotly.express as px  # Для быстрого создания графиков
from plotly.subplots import make_subplots  # Для создания subplots

# Scikit-learn - разделение данных
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    ShuffleSplit,
    cross_validate,
)

# Scikit-learn - предобработка
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.compose import TransformedTargetRegressor, ColumnTransformer
from sklearn.pipeline import Pipeline

# Scikit-learn - модели (ансамбли)
from sklearn.ensemble import (
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)

# Scikit-learn - метрики
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    PredictionErrorDisplay,
)

# Scikit-learn - анализ важности признаков
from sklearn.inspection import permutation_importance

# Scipy - для RandomizedSearchCV (распределения)
from scipy import stats
from scipy.stats import randint, uniform

# Утилиты
import time
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("✅ Все библиотеки успешно импортированы")

# ============================================================================
# Глобальный список для хранения результатов всех моделей
# ============================================================================
MODEL_RESULTS = []

def add_result(model_name, features_desc, model, X_train, y_train, X_test, y_test):
    """Добавляет результаты модели в глобальный список MODEL_RESULTS.
    
    Параметры:
    ----------
    model_name : str
        Название модели для отображения
    features_desc : str
        Описание используемых признаков
    model : estimator
        Обученная модель
    X_train : DataFrame
        Тренировочные признаки
    y_train : Series
        Тренировочные метки
    X_test : DataFrame
        Тестовые признаки
    y_test : Series
        Тестовые метки
    """
    # Предикты
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Метрики
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mae = mean_absolute_error(y_test, y_test_pred)
    
    # Сохраняем
    MODEL_RESULTS.append({
        'Model': model_name,
        'Features': features_desc,
        'R² Train': r2_train,
        'R² Test': r2_test,
        'Gap': r2_train - r2_test,
        'RMSE': rmse,
        'MAE': mae
    })
    
print("✅ Функция add_result() создана")


✅ Все библиотеки успешно импортированы
✅ Функция add_result() создана


# I. Выбор и подготовка набора данных с кратким EDA

EDA полностью копирует оный из работы по регрессии. Пайплайн взят оттуда же.

In [141]:
# ============================================================================
# ЗАГРУЗКА ДАННЫХ
# ============================================================================

data_path = Path('data/wine-quality/winequality-red.csv')
df = pd.read_csv(data_path)

print(f"Размерность: {df.shape}")
print(f"Признаки: {list(df.columns)}")
print("Целевая переменная: quality")


Размерность: (1599, 12)
Признаки: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
Целевая переменная: quality


In [142]:
# Предпросмотр данных после первичной загрузки
target_column = 'quality'
df.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [143]:
# Создаем subplot для гистограмм
feature_cols = df.drop(columns=[target_column]).columns
n_features = len(feature_cols)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols


In [144]:
fig2 = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=feature_cols,
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

for idx, col in enumerate(feature_cols):
    row = idx // n_cols + 1
    col_idx = idx % n_cols + 1
    
    fig2.add_trace(
        go.Box(
            y=df[col],
            name=col,
            showlegend=False,
            marker_color='crimson'
        ),
        row=row, col=col_idx
    )

fig2.update_layout(
    title={
        'text': 'Выбросы в числовых признаках (Box plots)',
        'x': 0.5,
        'xanchor': 'center'
    },
    height=500 * n_rows,
    width=1000,
    template='plotly_white'
)

fig2.show()

### Ссылка на предыдущий EDA

Детальный анализ данных был выполнен в работе [`lab/fourth_regression/part_2_wine.ipynb`](../fourth_regression/part_2_wine.ipynb). Ниже приведено краткое резюме ключевых выводов:

- Все 11 признаков числовые, пропусков нет.
- Целевая переменная `quality` принимает значения от 3 до 8.
- Сильные связи с целевой: alcohol (≈0.48), volatile acidity (≈-0.39), sulphates (≈0.25).
- Сильная мультиколлинеарность между некоторыми признаками, обоснованная естественными химическими законами (подробней в EDA лабораторной по регрессии).
- Дубликаты присутствуют - они оставлены, т.к. несут в себе полезную информацию и не являются случайными ошибками, а записей изначально не так много, чтобы расбрасываться ими.
- Признаки демонстрируют ассиметрию, многие имеют большую разницу медлу 75-перцентилем и максимумом - указывает потенциальное наличие выбросов.

**Итоговый результат EDA:** требуется провести стандартизацию признаков (химические признаки разного масштаба) и преобразование из-за наличия выбросов. Мной приведен именно график боксплотов для демонстрации выбросов (он уже демонстрировался в работе по регрессии) - я это сделал снова по той причине, что обработка выбросов, на мой взгляд, необязательна при использовании бустингов и других ансамблей на основе деревье решений, в целом как и стандартизация, так как сами деревья разобьют пространство признаков по порогам как им нужно. Я это делаю, потому что *хочу сравнить результат работы моделей в этой работе с результатми более простых моделей для решения регрессии в максимально одинаковых условиях.*

# II. Предварительная обработка данных

Построим pipeline предобработки. Сначала разделим данные по выборкам во избежании проблемы утечки данных.

In [145]:
# Разделение данных на матрицу признаков X и целевую переменную y
X = df.drop(columns=[target_column])
y = df[target_column]

print(f"Количество признаков: {X.shape[1]}")
print(f"\nСписок признаков:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("\nРазделение данных на train/test (ДО предобработки):")
print("="*80)
print(f"Размер тренировочной выборки (X_train): {X_train.shape}")
print(f"Размер тестовой выборки (X_test): {X_test.shape}")
print(f"\nСоотношение train/test: {X_train.shape[0]}/{X_test.shape[0]} = {X_train.shape[0]/X_test.shape[0]:.2f}:1")

Количество признаков: 11

Список признаков:
   1. fixed acidity
   2. volatile acidity
   3. citric acid
   4. residual sugar
   5. chlorides
   6. free sulfur dioxide
   7. total sulfur dioxide
   8. density
   9. pH
  10. sulphates
  11. alcohol

Разделение данных на train/test (ДО предобработки):
Размер тренировочной выборки (X_train): (1279, 11)
Размер тестовой выборки (X_test): (320, 11)

Соотношение train/test: 1279/320 = 4.00:1


С прошлой работы я принял решение изменить Pipeline - я пошел трудным путем, применяя логарифмическое преобразование к избранным признакам (а самый важный признак по корреляции с целевой алкоголь я забыл преобразовать) с высокой правосторонней ассиметрией. В этот раз, я решил просто применить PowerTransformer для всех признаков - он применит сразу и преобразование и стандартизацию, то что нужно, причем, он сам подберет лямбда для силы преобразования признака.

In [146]:
# Построение Pipeline с использованием PowerTransformer (Yeo-Johnson)
print("="*80)
print("Построение Pipeline предобработки:")
print("="*80)

# Все числовые признаки подвергаются Yeo-Johnson с одновременной стандартизацией
feature_order = list(X_train.columns)

# Создание ColumnTransformer: один шаг PowerTransformer для всех признаков
preprocessor = ColumnTransformer(
    transformers=[
        ('power_transform', PowerTransformer(method='yeo-johnson', standardize=True), feature_order)
    ],
    remainder='drop'
)

# Pipeline состоит только из предобработки (PowerTransformer уже стандартизует данные)
preprocessing_pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

# Применение предобработки к тренировочным и тестовым данным
X_train_final = preprocessing_pipeline.fit_transform(X_train)
X_test_final = preprocessing_pipeline.transform(X_test)

# Преобразование обратно в DataFrame для удобства
X_train_final = pd.DataFrame(X_train_final, columns=feature_order, index=X_train.index)
X_test_final = pd.DataFrame(X_test_final, columns=feature_order, index=X_test.index)

print("Pipeline создан и применен (PowerTransformer, Yeo-Johnson)")
print(f"\nРазмерность данных после предобработки:")
print(f"  X_train_final: {X_train_final.shape}")
print(f"  X_test_final: {X_test_final.shape}")

# Выбираем несколько признаков для визуализации
sample_features = ['alcohol', 'volatile acidity', 'sulphates', 'pH']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{feat} - до и после' for feat in sample_features],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

for idx, feat in enumerate(sample_features):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    # До предобработки
    fig.add_trace(
        go.Histogram(
            x=X_train[feat],
            name='До',
            marker_color='steelblue',
            opacity=0.6,
            nbinsx=30,
            showlegend=(idx == 0)
        ),
        row=row, col=col
    )
    
    # После предобработки
    fig.add_trace(
        go.Histogram(
            x=X_train_final[feat],
            name='После',
            marker_color='crimson',
            opacity=0.6,
            nbinsx=30,
            showlegend=(idx == 0)
        ),
        row=row, col=col
    )

fig.update_layout(
    title={
        'text': 'Распределение признаков до и после предобработки (PowerTransformer, Yeo-Johnson)',
        'x': 0.5,
        'xanchor': 'center'
    },
    height=800,
    width=1000,
    template='plotly_white',
    barmode='overlay'
)

fig.show()

Построение Pipeline предобработки:
Pipeline создан и применен (PowerTransformer, Yeo-Johnson)

Размерность данных после предобработки:
  X_train_final: (1279, 11)
  X_test_final: (320, 11)


In [147]:
# Распределение всех признаков после пайплайна
print("="*80)
print("Визуализация распределения всех признаков после предобработки:")
print("="*80)

# Получаем все признаки из X_train_final
all_features = X_train_final.columns.tolist()
n_features = len(all_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

# Создаем subplot для гистограмм всех признаков
fig_dist = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=all_features,
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

# Добавляем гистограммы для каждого признака
for idx, feat in enumerate(all_features):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    
    fig_dist.add_trace(
        go.Histogram(
            x=X_train_final[feat],
            name=feat,
            marker_color='steelblue',
            opacity=0.7,
            nbinsx=40,
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Добавляем вертикальную линию для среднего значения
    mean_val = X_train_final[feat].mean()
    fig_dist.add_vline(
        x=mean_val,
        line_dash="dash",
        line_color="crimson",
        row=row, col=col
    )

fig_dist.update_layout(
    title={
        'text': 'Распределение всех признаков после предобработки (Log + StandardScaler)',
        'x': 0.5,
        'xanchor': 'center'
    },
    height=400 * n_rows,
    width=1200,
    template='plotly_white'
)

fig_dist.show()

# Вывод статистики по всем признакам
print("\n" + "-"*80)
print("Статистика по всем признакам после предобработки:")
print("-"*80)
stats_df = pd.DataFrame({
    'Признак': all_features,
    'Среднее': [X_train_final[feat].mean() for feat in all_features],
    'Стд. отклонение': [X_train_final[feat].std() for feat in all_features],
    'Мин': [X_train_final[feat].min() for feat in all_features],
    'Макс': [X_train_final[feat].max() for feat in all_features],
    'Медиана': [X_train_final[feat].median() for feat in all_features]
})
print(stats_df.to_string(index=False))

Визуализация распределения всех признаков после предобработки:



--------------------------------------------------------------------------------
Статистика по всем признакам после предобработки:
--------------------------------------------------------------------------------
             Признак       Среднее  Стд. отклонение       Мин     Макс   Медиана
       fixed acidity -1.597193e-16         1.000391 -3.446507 2.825925 -0.103981
    volatile acidity  3.388828e-16         1.000391 -3.048843 3.757834  0.048958
         citric acid  2.680507e-16         1.000391 -1.531984 2.861413  0.034848
      residual sugar -5.291571e-15         1.000391 -4.947541 2.538991 -0.024507
           chlorides -3.749932e-16         1.000391 -6.495669 2.896491  0.006904
 free sulfur dioxide -8.333183e-18         1.000391 -2.940845 2.722660  0.081561
total sulfur dioxide -6.795016e-16         1.000391 -2.409841 3.038277  0.029868
             density -1.414281e-11         1.000391 -3.694206 3.646070  0.020539
                  pH -3.355495e-15         1.000391 -4.087

# III. Обучение сложных моделей (до 2 баллов)

## 1. Метрики для сравнения моделей

**Выбранные метрики:**
- **R² (коэффициент детерминации)** — основная метрика
- **RMSE (Root Mean Squared Error)** — корень из среднеквадратичной ошибки
- **MAE (Mean Absolute Error)** — средняя абсолютная ошибка

**Обоснование выбора метрик:**

1. **R² (коэффициент детерминации)** — основная метрика для регрессии
   - Показывает долю дисперсии целевой переменной, объясненную моделью
   - Значения от -∞ до 1 (чем ближе к 1, тем лучше)
   - Интерпретируемая метрика: показывает, насколько модель лучше простого среднего
   - Используется как основная метрика для сравнения моделей
   - **Почему важна:** Позволяет понять общую способность модели объяснять вариативность данных

2. **RMSE (Root Mean Squared Error)** — метрика в единицах целевой переменной
   - Вычисляется как квадратный корень из MSE: `RMSE = √(MSE)`
   - Имеет те же единицы измерения, что и целевая переменная (для wine quality — баллы)
   - Более чувствительна к большим ошибкам (штрафует выбросы)
   - **Почему важна:** Позволяет интерпретировать ошибку в естественных единицах (например, "модель ошибается в среднем на 0.5 балла")

3. **MAE (Mean Absolute Error)** — устойчивая к выбросам метрика
   - Вычисляется как среднее абсолютных значений ошибок: `MAE = mean(|y_true - y_pred|)`
   - Менее чувствительна к выбросам, чем RMSE
   - Также имеет единицы целевой переменной
   - **Почему важна:** Показывает среднюю величину ошибки без учета квадратичного штрафа, что полезно при наличии выбросов в данных

**Дополнительные метрики (опционально):**
- **OOB Score (Out-of-Bag)** — для Random Forest, позволяет оценить качество без отдельной валидационной выборки
- **Training History** — для бустингов, показывает процесс обучения и возможное переобучение

**Почему именно этот набор метрик:**
- **R²** дает общую оценку качества модели
- **RMSE** показывает ошибку в естественных единицах и чувствительна к большим ошибкам
- **MAE** дополняет RMSE, показывая среднюю ошибку без квадратичного штрафа
- Вместе эти три метрики дают полную картину качества модели с разных точек зрения

In [148]:
def calculate_metrics(y_true, y_pred, model_name="Model", y_train_true=None, y_train_pred=None):
    """Расчет и вывод метрик для регрессии
    
    Параметры:
    ----------
    y_true : array-like
        Истинные значения для тестовой выборки
    y_pred : array-like
        Предсказанные значения для тестовой выборки
    model_name : str
        Имя модели для вывода
    y_train_true : array-like, optional
        Истинные значения для тренировочной выборки (для сравнения)
    y_train_pred : array-like, optional
        Предсказанные значения для тренировочной выборки (для сравнения)
    """
    # Метрики на тестовой выборке
    r2_test = r2_score(y_true, y_pred)
    rmse_test = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_test = mean_absolute_error(y_true, y_pred)
    
    print(f"\n{model_name}:")
    print(f"{'Метрика':<20} {'Тренировочный набор':<25} {'Тестовый набор':<20}")
    print("-" * 65)
    
    # Если переданы train данные, вычисляем метрики и для них
    if y_train_true is not None and y_train_pred is not None:
        r2_train = r2_score(y_train_true, y_train_pred)
        rmse_train = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
        mae_train = mean_absolute_error(y_train_true, y_train_pred)
        
        print(f"{'R²':<20} {r2_train:<25.4f} {r2_test:<20.4f}")
        print(f"{'RMSE':<20} {rmse_train:<25.4f} {rmse_test:<20.4f}")
        print(f"{'MAE':<20} {mae_train:<25.4f} {mae_test:<20.4f}")
        
        # Разница между train и test R² (для оценки переобучения)
        r2_diff = r2_train - r2_test
        print(f"\nРазница R² (train - test): {r2_diff:.4f}")
        if r2_diff > 0.1:
            print("⚠️ Возможное переобучение (разница > 0.1)")
        else:
            print("✅ Переобучение незначительное (разница <= 0.1)")
        
        return {
            'train': {'r2': r2_train, 'rmse': rmse_train, 'mae': mae_train},
            'test': {'r2': r2_test, 'rmse': rmse_test, 'mae': mae_test},
            'r2_diff': r2_diff
        }
    else:
        # Если train данные не переданы, выводим только test метрики
        print(f"{'R²':<20} {'N/A':<25} {r2_test:<20.4f}")
        print(f"{'RMSE':<20} {'N/A':<25} {rmse_test:<20.4f}")
        print(f"{'MAE':<20} {'N/A':<25} {mae_test:<20.4f}")
        
        return {'r2': r2_test, 'rmse': rmse_test, 'mae': mae_test}

Функция для графиков метрик

In [149]:
def plot_model_report_separate(
    metrics: dict,
    feature_importance: pd.DataFrame | None,
    model_name: str,
    width_metrics: int = 800,
    height_metrics: int = 500,
    width_fi: int = 800,
    height_fi: int = 600,
):
    """Построение двух отдельных графиков: метрики train/test и важность признаков.

    Параметры
    ----------
    metrics : dict
        Словарь метрик вида:
        {
            'train': {'r2': float, 'rmse': float, 'mae': float},
            'test':  {'r2': float, 'rmse': float, 'mae': float}
        }.
        Предполагается, что значения уже посчитаны (например, через calculate_metrics).

    feature_importance : pandas.DataFrame | None
        Таблица важностей признаков с колонками ['feature', 'importance'].
        Если None или пусто, график важностей не строится.

    model_name : str
        Название модели для заголовков графиков (например, "Random Forest (Baseline)").

    width_metrics : int, optional
        Ширина графика метрик в пикселях (по умолчанию 800).

    height_metrics : int, optional
        Высота графика метрик в пикселях (по умолчанию 500).

    width_fi : int, optional
        Ширина графика важностей в пикселях (по умолчанию 800).

    height_fi : int, optional
        Высота графика важностей в пикселях (по умолчанию 600).

    Возвращает
    ----------
    tuple[go.Figure, go.Figure | None]
        (fig_metrics, fig_fi), где fig_metrics — график метрик,
        fig_fi — график важностей или None, если важности не переданы.

    Поведение
    ----------
    - Строит отдельный bar-chart для метрик (Train/Test).
    - Строит отдельный горизонтальный bar-chart для важностей признаков,
      без подписей чисел на столбцах.
    - Оба графика отображаются через fig.show(). Если важностей нет,
      отображается только график метрик.
    """
    # Метрики (отдельный Figure)
    metrics_df = pd.DataFrame({
        'Метрика': ['R²', 'RMSE', 'MAE'],
        'Train': [metrics['train']['r2'], metrics['train']['rmse'], metrics['train']['mae']],
        'Test':  [metrics['test']['r2'],  metrics['test']['rmse'],  metrics['test']['mae']],
    })

    fig_metrics = go.Figure()
    fig_metrics.add_trace(go.Bar(
        name='Train',
        x=metrics_df['Метрика'],
        y=metrics_df['Train'],
        marker_color='steelblue',
        text=[f'{v:.4f}' for v in metrics_df['Train']],
        textposition='auto'
    ))
    fig_metrics.add_trace(go.Bar(
        name='Test',
        x=metrics_df['Метрика'],
        y=metrics_df['Test'],
        marker_color='crimson',
        text=[f'{v:.4f}' for v in metrics_df['Test']],
        textposition='auto'
    ))
    fig_metrics.update_layout(
        title={'text': f'Метрики качества: {model_name}', 'x': 0.5, 'xanchor': 'center'},
        xaxis_title='Метрика',
        yaxis_title='Значение',
        barmode='group',
        template='plotly_white',
        width=width_metrics,
        height=height_metrics,
        legend_title_text="Split"
    )

    fig_fi = None
    if feature_importance is not None and len(feature_importance) > 0:
        fi = feature_importance.copy().sort_values("importance", ascending=True)
        fig_fi = go.Figure()
        fig_fi.add_trace(go.Bar(
            x=fi['importance'],
            y=fi['feature'],
            orientation='h',
            marker_color='coral'
            # подписи чисел на столбцах не выводим
        ))
        fig_fi.update_layout(
            title={'text': 'Важность признаков', 'x': 0.5, 'xanchor': 'center'},
            xaxis_title='Важность',
            yaxis_title='Признак',
            template='plotly_white',
            width=width_fi,
            height=height_fi,
            showlegend=False
        )

    fig_metrics.show()
    if fig_fi is not None:
        fig_fi.show()

    return None

## 2. Выбор моделей

**Выбранные модели:**
1. **Random Forest (RandomForestRegressor)** — ансамбль деревьев решений на основе бэггинга
2. **GradientBoostingRegressor** — классический градиентный бустинг
3. **HistGradientBoostingRegressor** — оптимизированный градиентный бустинг на основе гистограмм

**Обоснование выбора:**

1. **Random Forest:**
   - Простой и эффективный ансамбль на основе бэггинга
   - Устойчив к переобучению благодаря случайности
   - Имеет встроенную оценку OOB (Out-of-Bag), что позволяет оценить качество без валидационной выборки
   - Хорошо работает "из коробки" без тщательной настройки

2. **GradientBoostingRegressor:**
   - Классический градиентный бустинг от scikit-learn
   - Последовательное обучение деревьев с коррекцией ошибок
   - Позволяет отслеживать историю обучения через `train_score_`
   - Хорошо работает на структурированных данных

3. **HistGradientBoostingRegressor:**
   - Современная оптимизированная версия градиентного бустинга
   - Использует гистограммы для ускорения обучения
   - Обычно быстрее и эффективнее GradientBoostingRegressor
   - Поддерживает категориальные признаки (хотя в нашем случае все признаки числовые)

**План сравнения:**
1. Обучить все три модели с базовыми параметрами на тренировочной выборке
2. Оценить метрики на тестовой выборке
3. Визуализировать важность признаков для каждой модели
4. Сравнить результаты и улучшить выбранные модели


## 3. Обучение Random Forest (базовая модель)

**Цель:** Посмотреть как модель работает "из коробки" используя тренировочную выборку


In [150]:
# Базовая модель Random Forest
rf_baseline = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    oob_score=True  # Out-of-bag оценка
)

# Обучение на тренировочной выборке
print("Обучение Random Forest...")
rf_baseline.fit(X_train_final, y_train)

# Предсказания на тренировочной и тестовой выборках
y_train_pred_rf = rf_baseline.predict(X_train_final)
y_pred_rf = rf_baseline.predict(X_test_final)

# Метрики с сравнением train/test
metrics_rf = calculate_metrics(
    y_test, y_pred_rf, 
    "Random Forest (Baseline)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_rf
)

print(f"OOB Score: {rf_baseline.oob_score_:.4f}")

# Важность признаков
feature_importance_rf = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': rf_baseline.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (топ-5):")
print(feature_importance_rf.head())

# Добавляем результат в глобальный список
add_result("RF Baseline", "All (11)", rf_baseline, X_train_final, y_train, X_test_final, y_test)


Обучение Random Forest...

Random Forest (Baseline):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.9265                    0.5404              
RMSE                 0.2187                    0.5481              
MAE                  0.1541                    0.4202              

Разница R² (train - test): 0.3861
⚠️ Возможное переобучение (разница > 0.1)
OOB Score: 0.4556

Важность признаков (топ-5):
                 feature  importance
10               alcohol    0.270868
9              sulphates    0.148406
1       volatile acidity    0.111547
6   total sulfur dioxide    0.076786
4              chlorides    0.071132


In [151]:
plot_model_report_separate(metrics_rf, feature_importance_rf, model_name="Random Forest (Baseline)")

Модель демонстрирует явное переобучение (огромная разница в R^2). Показатель OOB Score является встроенным методом валидации для случайного леса, использующим объекты не попавшие в бутстрап выборку при обучении деревье - то что он ниже чем r^2 на тесте еще раз подтверждает слабую предсказательную способность на данных, которые модель не видела. Ошибки RMSE и MAE говорят о той же проблеме.

Модель выделила алкоголь, сульфаты и переменную кислотность как значимые для результата предсказания факторы.

Проблема этой модели скорее всего в том, что нужно ограничить глубину деревье - это вызывает такое переобучение.

## 4. Обучение GradientBoostingRegressor (базовая модель)


In [152]:
# Базовая модель GradientBoosting
gb_baseline = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Обучение на тренировочной выборке
print("Обучение GradientBoostingRegressor...")
gb_baseline.fit(X_train_final, y_train)

# Предсказания на тренировочной и тестовой выборках
y_train_pred_gb = gb_baseline.predict(X_train_final)
y_pred_gb = gb_baseline.predict(X_test_final)

# Метрики с сравнением train/test
metrics_gb = calculate_metrics(
    y_test, y_pred_gb, 
    "Gradient Boosting (Baseline)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_gb
)

# Важность признаков
feature_importance_gb = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': gb_baseline.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (топ-5):")
print(feature_importance_gb.head())

# Добавляем результат в глобальный список
add_result("GB Baseline", "All (11)", gb_baseline, X_train_final, y_train, X_test_final, y_test)


Обучение GradientBoostingRegressor...

Gradient Boosting (Baseline):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.6173                    0.4414              
RMSE                 0.4990                    0.6042              
MAE                  0.3873                    0.4856              

Разница R² (train - test): 0.1759
⚠️ Возможное переобучение (разница > 0.1)

Важность признаков (топ-5):
                 feature  importance
10               alcohol    0.375965
9              sulphates    0.194192
1       volatile acidity    0.125359
6   total sulfur dioxide    0.075183
0          fixed acidity    0.056945


In [153]:
plot_model_report_separate(metrics_gb, feature_importance_gb, model_name="Gradient Boosting Regressor (Baseline)")

In [154]:
# История обучения GradientBoosting: Train vs Test RMSE
train_rmse_curve = []
test_rmse_curve = []

for y_pred_tr, y_pred_te in zip(
    gb_baseline.staged_predict(X_train_final),
    gb_baseline.staged_predict(X_test_final)
):
    train_rmse_curve.append(np.sqrt(mean_squared_error(y_train, y_pred_tr)))
    test_rmse_curve.append(np.sqrt(mean_squared_error(y_test, y_pred_te)))

best_iter = int(np.argmin(test_rmse_curve))
print(f"Лучшее RMSE на тесте: итерация {best_iter}, RMSE={test_rmse_curve[best_iter]:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(train_rmse_curve))),
    y=train_rmse_curve,
    mode='lines+markers',
    name='Train RMSE',
    line=dict(color='steelblue', width=2),
    marker=dict(size=4)
))
fig.add_trace(go.Scatter(
    x=list(range(len(test_rmse_curve))),
    y=test_rmse_curve,
    mode='lines+markers',
    name='Test RMSE',
    line=dict(color='crimson', width=2),
    marker=dict(size=4)
))
fig.add_vline(x=best_iter, line=dict(color='gray', width=1, dash='dash'), annotation_text='min test RMSE', annotation_position='top')
fig.update_layout(
    title='GradientBoosting Training vs Test RMSE',
    xaxis_title='Iteration',
    yaxis_title='RMSE',
    template='plotly_white',
    height=500,
    width=900
)
fig.show()


Лучшее RMSE на тесте: итерация 88, RMSE=0.6042


Модель показала себя хуже случайного леса, но при этом и переобучение не такое большое, что видно по разнице r^2. В качестве наиболее влияющих на результат признаков выделене те же, что и случайным лесом - очередной намек на то, что следует обратить внимание на них при попытке инженерии признаков (например взять количество алкоголя и сульфатов и сделать из них полиномиальные признаки или биннинг) или стоит проверить как они преобразуются.

По графику обучения видно, что на тестовой выборке величина ошибка перестает падать после 35-40 итераций.

Вообще, алгоритм градиентного бустинга оказался чувствительным к выбросам, что заставило меня задаться в очередной раз вопрос - не стоит ли изменить подход к преобразованию признаков, т.к. вероятно логарифмическое преобразование не избранных признаках не дает хорошего результата.

Проблема связана не с характером распределения (деревьям на это почти все равно), а с количеством выбросов - для улучшения результатов можно попробовать использовать например QuantileTransformer для агрессивного избавления от шумов на избранных признаках с большим количеством выбросов.

## 5. Сравнение базовых моделей


In [155]:
# Сравнение базовых моделей
comparison_baseline = pd.DataFrame({
    'Model': ['Random Forest', 'GradientBoosting'],
    'R²': [
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_gb)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_gb))
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_gb)
    ]
})

print("="*80)
print("Сравнение базовых моделей на тестовой выборке:")
print("="*80)
print(comparison_baseline.to_string(index=False))


Сравнение базовых моделей на тестовой выборке:
           Model       R²     RMSE      MAE
   Random Forest 0.540354 0.548070 0.420250
GradientBoosting 0.441400 0.604192 0.485558


In [156]:
# Визуализация сравнения метрик
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('R² Score', 'RMSE', 'MAE'),
    horizontal_spacing=0.1
)

# R² Score
fig.add_trace(
    go.Bar(
        x=comparison_baseline['Model'],
        y=comparison_baseline['R²'],
        name='R²',
        marker_color='steelblue',
        text=[f'{val:.4f}' for val in comparison_baseline['R²']],
        textposition='auto'
    ),
    row=1, col=1
)

# RMSE
fig.add_trace(
    go.Bar(
        x=comparison_baseline['Model'],
        y=comparison_baseline['RMSE'],
        name='RMSE',
        marker_color='coral',
        text=[f'{val:.4f}' for val in comparison_baseline['RMSE']],
        textposition='auto',
        showlegend=False
    ),
    row=1, col=2
)

# MAE
fig.add_trace(
    go.Bar(
        x=comparison_baseline['Model'],
        y=comparison_baseline['MAE'],
        name='MAE',
        marker_color='lightgreen',
        text=[f'{val:.4f}' for val in comparison_baseline['MAE']],
        textposition='auto',
        showlegend=False
    ),
    row=1, col=3
)

fig.update_layout(
    title_text='Сравнение базовых моделей на тесте',
    height=500,
    width=1200,
    template='plotly_white',
    showlegend=False
)

fig.update_xaxes(tickangle=-45)
fig.show()


## 6. Выводы о базовых моделях


Сравнивая модели, одназначно лучше себя показывает random forest - выше процент объясненных значений, ниже ошибки. При этом, модель показывает феноменально огромное переобучение - вероятно это проблема отсутствия ограчниения на глубину дерева, следовательно модель просто все заучила. По OOB score модель приблизительно на уровне r2 обычного бустинга, но ошибки все равно ниже.

Стоит отметить, что обе модели единогласно выделили наиболее важные признаки - процент алкоголя, сульфаты и volatile accidity, так же в топ-5 для RF находятся хлориды.

В следующей главе попробуем оптимизировать обе модели и повысить точность, а главное - избавиться от переобучения.

# IV. Оптимизация выбранных моделей (до 3 баллов)

**Требование:** Оптимизировать выбранные модели, указать какие гиперпараметры позволили достичь хороших результатов.

**Схема оптимизации:**
1. GridSearchCV на всех признаках с обязательным ограничением `max_depth` и `min_samples_leaf` — это даст «золотой стандарт» качества
2. Анализ Feature Importance на оттюнингованной модели (важность признаков у переобученной модели может быть ложной)
3. Отсечение признаков по порогу важности (например, 0.02 или 2%), а не по количеству
4. Сравнение: если модель на усеченных признаках дает результат не хуже (или всего на 1% хуже), чем на полных — оставляйте усеченную

## 1. Выбор моделей для оптимизации

Для оптимизации выбраны **Random Forest** и **GradientBoosting**:
- Random Forest показал лучший результат среди базовых моделей (R² = 0.5404)
- GradientBoosting показал второй результат (R² = 0.4414) и имеет потенциал для улучшения через тюнинг гиперпараметров
- Обе модели основаны на деревьях решений и хорошо подходят для данной задачи регрессии

## 2. Оптимизация Random Forest

### 2.1 GridSearchCV на всех признаках, анализ важности и визуализация (золотой стандарт)

In [157]:
# Пространство гиперпараметров для Random Forest
# ВАЖНО: max_depth и min_samples_leaf должны быть ограничены (не None для max_depth)

# 1. Определяем стратегию кросс-валидации
cv_strategy = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# 2. Задаем сетку параметров
param_grid_rf = {
    'n_estimators': [200, 300, 400, 500, 600, 800, 1200],
    'max_depth': [12, 14, 15, 16, 17, 18],  # Обязательное ограничение (не None)
    'min_samples_split': [2, 3, 4, 5, 6],
    'min_samples_leaf': [4, 6, 8, 10, 12],  # Обязательное ограничение
    'max_features': ['sqrt', 'log2', 0.2, 0.3, 0.4, 0.5],
    'bootstrap': [True] # Использовать ли Bootstrap - пробуем уменьшить переобучение
}

# вторая сетка, использую ее, чтобы попробовать поборот голодание модели
param_grid_rf_bridge = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [12, 13, 14, 16], 
    'min_samples_leaf': [6], 
    'min_samples_split': [12],
    'max_features': [0.2, 0.3, 0.4, 0.5, 0.6, 'sqrt'], 
    'bootstrap': [True]
}

# GridSearchCV на всех признаках
# Используем X_train_final и y_train для кросс-валидации
# 3. Запускаем поиск
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_grid_rf_bridge,
    cv=cv_strategy,
    scoring='r2',
    n_jobs=-1, # распаралеллить вычисления по потокам проца
    return_train_score=True, # Чтобы сравнить Train/Test прямо в результатах поиска,
    verbose=2
)

print("Запуск GridSearchCV для Random Forest...")
rf_grid.fit(X_train_final, y_train)

# 4. Извлечение лучшей модели и результатов
print("\n" + "="*80)
print("Результаты оптимизации Random Forest:")
print("="*80)
print(f"Лучшие параметры: {rf_grid.best_params_}")

rf_optimized = rf_grid.best_estimator_

# Предикты
y_train_pred = rf_optimized.predict(X_train_final)
y_test_pred = rf_optimized.predict(X_test_final)

# 5. Оценка переобучения после оптимизации
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
gap = r2_train - r2_test

print(f"R² на тренировочном наборе: {r2_train:.4f}")
print(f"R² на тестовом наборе:      {r2_test:.4f}")
print(f"Разница (Gap):              {gap:.4f}")

# Добавляем результат в глобальный список
add_result("RF Optimized (All)", "All (11)", rf_optimized, X_train_final, y_train, X_test_final, y_test)

Запуск GridSearchCV для Random Forest...
Fitting 5 folds for each of 96 candidates, totalling 480 fits

Результаты оптимизации Random Forest:
Лучшие параметры: {'bootstrap': True, 'max_depth': 16, 'max_features': 0.6, 'min_samples_leaf': 6, 'min_samples_split': 12, 'n_estimators': 400}
R² на тренировочном наборе: 0.6964
R² на тестовом наборе:      0.5056
Разница (Gap):              0.1908


В итоге, перебирая гиперпараметры как в том меме с Карлсоном, я решил пожертвовать 4 процентам точности предсказания на тесте в пользу уменьшения разницы точности на 20 пунктов. Гэп в 0.19 уже не выглядит так страшно, как 0.38. При переборе (в коде можно увидеть как примерно выглядела сетка параметров изначально и к чему я пришел) я столкнулся с тем, что гридсерч ориентируется именно на максимизацию тестовой точности - у меня получалась точность на тесте в районе 0.54, как и в бейзлайн модели, при этом точность на трейне была в районе 0.85-0.8 - меня это не устраивало, хотелось, как сказал вышел, бороться с переобучением в первую очередь.

По поводу гиперпараметров:
- min_samples_leaf=6 - гридсерч упорно выбирал наименьшее значение и стремился заучить паттерн, при этом, увеличение грубости позволило уверенее бороться с переобучением
- max_features=0.6 - стандартно модель выбирает около 30 процентов признаков, тут же пришлось брать больше из-за увеличенного размера листьев

Попробую глянуть на признаки и обучить модель только на топ-5 по важности

In [158]:
# Анализ важности признаков на оттюнингованной модели
feature_importance_rf = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': rf_optimized.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + "="*80)
print("Важность признаков (оттюнингованная модель):")
print("="*80)
print(feature_importance_rf.to_string(index=False))


Важность признаков (оттюнингованная модель):
             feature  importance
             alcohol    0.299285
           sulphates    0.184232
    volatile acidity    0.130348
total sulfur dioxide    0.074064
             density    0.055904
           chlorides    0.048796
         citric acid    0.048336
       fixed acidity    0.044989
                  pH    0.043551
 free sulfur dioxide    0.037615
      residual sugar    0.032879


In [159]:
# Метрики для визуализации
metrics_rf_opt = {
    'train': {
        'r2': r2_score(y_train, y_train_pred),
        'rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'mae': mean_absolute_error(y_train, y_train_pred)
    },
    'test': {
        'r2': r2_score(y_test, y_test_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'mae': mean_absolute_error(y_test, y_test_pred)
    }
}

print("\n" + "="*80)
print("Метрики Random Forest (Optimized, all features):")
print("="*80)
print(f"Train - R²: {metrics_rf_opt['train']['r2']:.4f}, RMSE: {metrics_rf_opt['train']['rmse']:.4f}, MAE: {metrics_rf_opt['train']['mae']:.4f}")
print(f"Test  - R²: {metrics_rf_opt['test']['r2']:.4f}, RMSE: {metrics_rf_opt['test']['rmse']:.4f}, MAE: {metrics_rf_opt['test']['mae']:.4f}")


# Визуализация метрик и важности признаков
plot_model_report_separate(
    metrics=metrics_rf_opt,
    feature_importance=feature_importance_rf,
    model_name="Random Forest (Optimized, all features)"
)

# Определение порога для отсечения (например, 2% или 0.02)
importance_threshold = 0.05  # 2%
print(f"\nПорог важности: {importance_threshold}")

# Признаки выше порога
selected_features_rf = feature_importance_rf[
    feature_importance_rf['importance'] >= importance_threshold
]['feature'].tolist()

print(f"Количество признаков выше порога: {len(selected_features_rf)}%")
print(f"Отобранные признаки: {selected_features_rf}")
print(f"Отброшенные признаки: {set(X_train_final.columns) - set(selected_features_rf)}")



Метрики Random Forest (Optimized, all features):
Train - R²: 0.6964, RMSE: 0.4444, MAE: 0.3268
Test  - R²: 0.5056, RMSE: 0.5684, MAE: 0.4525



Порог важности: 0.05
Количество признаков выше порога: 5%
Отобранные признаки: ['alcohol', 'sulphates', 'volatile acidity', 'total sulfur dioxide', 'density']
Отброшенные признаки: {'pH', 'fixed acidity', 'chlorides', 'free sulfur dioxide', 'residual sugar', 'citric acid'}


### 2.2 Обучение Random Forest на усеченных признаках и сравнение

Я отобрал признаки, которые пробивают порог в 5 процентов влияния на целевую переменную. Попробуем подобрать гиперпараметры и обучить модель с ними и посмотретть на результат.

In [160]:
# 1. Стратегия кросс-валидации
cv_strategy = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# 2. Сетка параметров
# ВАЖНО: Для усеченного набора признаков (5 признаков) уменьшаем глубину,
# чтобы избежать переобучения
param_grid_rf_top = {
    'n_estimators': [200, 300],
    'max_depth': [5, 7, 9, 11],  # Меньше глубина, так как признаков мало
    'min_samples_leaf': [5, 10], 
    'max_features': ['sqrt', 1.0],  # 1.0 = использовать все 5 признаков
    'bootstrap': [True]
}

# 3. Запуск поиска (используем только selected_features_rf)
rf_grid_reduced = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_grid_rf_top,
    cv=cv_strategy,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

print(f"Запуск GridSearchCV для Random Forest ({len(selected_features_rf)} признаков)...")
rf_grid_reduced.fit(X_train_final[selected_features_rf], y_train)

# 4. Извлечение лучшей модели
# 4. Извлечение лучшей модели и результатов
print("\n" + "="*80)
print("Результаты оптимизации Random Forest на топ признаках:")
print("="*80)
print(f"Лучшие параметры: {rf_grid_reduced.best_params_}")
rf_reduced = rf_grid_reduced.best_estimator_

# Предикты (строго на выбранных признаках)
y_pred_rf_reduced_train = rf_reduced.predict(X_train_final[selected_features_rf])
y_pred_rf_reduced_test = rf_reduced.predict(X_test_final[selected_features_rf])

r2_train_red = r2_score(y_train, y_pred_rf_reduced_train)
r2_test_red = r2_score(y_test, y_pred_rf_reduced_test)
gap_red = r2_train_red - r2_test_red

print("\n" + "="*40)
print("АНАЛИЗ ПЕРЕОБУЧЕНИЯ (REDUCED MODEL)")
print("="*40)
print(f"R² на тренировочном наборе: {r2_train_red:.4f}")
print(f"R² на тестовом наборе:      {r2_test_red:.4f}")
print(f"Разница (Gap):              {gap_red:.4f}")

# Добавляем результат в глобальный список
add_result("RF Optimized (Reduced)", f"Top {len(selected_features_rf)}", rf_reduced, 
           X_train_final[selected_features_rf], y_train, 
           X_test_final[selected_features_rf], y_test)

Запуск GridSearchCV для Random Forest (5 признаков)...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

Результаты оптимизации Random Forest на топ признаках:
Лучшие параметры: {'bootstrap': True, 'max_depth': 11, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 300}

АНАЛИЗ ПЕРЕОБУЧЕНИЯ (REDUCED MODEL)
R² на тренировочном наборе: 0.6490
R² на тестовом наборе:      0.4659
Разница (Gap):              0.1831


In [161]:
# 5. Сбор метрик для статистики и визуализации
metrics_rf_reduced = {
    'train': {
        'r2': r2_score(y_train, y_pred_rf_reduced_train),
        'rmse': np.sqrt(mean_squared_error(y_train, y_pred_rf_reduced_train)),
        'mae': mean_absolute_error(y_train, y_pred_rf_reduced_train)
    },
    'test': {
        'r2': r2_score(y_test, y_pred_rf_reduced_test),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_rf_reduced_test)),
        'mae': mean_absolute_error(y_test, y_pred_rf_reduced_test)
    }
}

# Важность признаков
feature_importance_rf_reduced = pd.DataFrame({
    'feature': selected_features_rf,
    'importance': rf_reduced.feature_importances_
}).sort_values('importance', ascending=False)

In [162]:
# 6. Визуализация (графики)
plot_model_report_separate(
    metrics=metrics_rf_reduced,
    feature_importance=feature_importance_rf_reduced,
    model_name=f"Random Forest (Optimized, {len(selected_features_rf)} features)"
)

# 7. Сравнение с "золотым стандартом" (полной моделью)
# Предполагается, что metrics_rf_opt уже рассчитана ранее для полной модели
r2_full = metrics_rf_opt['test']['r2']
r2_reduced = metrics_rf_reduced['test']['r2']
r2_diff = r2_full - r2_reduced
r2_diff_percent = (r2_diff / r2_full) * 100 if r2_full > 0 else 0

print("\n" + "="*80)
print("РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ И СРАВНЕНИЯ:")
print("="*80)
print(f"Лучшие параметры поиска: {rf_grid_reduced.best_params_}")
print(f"R² (все признаки):      {r2_full:.4f}")
print(f"R² (усеченные признаки): {r2_reduced:.4f}")
print(f"Разница R²:             {r2_diff:.4f} ({r2_diff_percent:.2f}%)")

rmse_full = metrics_rf_opt['test']['rmse']
rmse_reduced = metrics_rf_reduced['test']['rmse']
print(f"RMSE (все признаки):      {rmse_full:.4f}")
print(f"RMSE (усеченные признаки): {rmse_reduced:.4f}")


РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ И СРАВНЕНИЯ:
Лучшие параметры поиска: {'bootstrap': True, 'max_depth': 11, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 300}
R² (все признаки):      0.5056
R² (усеченные признаки): 0.4659
Разница R²:             0.0397 (7.85%)
RMSE (все признаки):      0.5684
RMSE (усеченные признаки): 0.5908


In [163]:
# 8. Визуализация сравнения метрик
comparison_data = pd.DataFrame({
    'Метрика': ['R²', 'RMSE', 'MAE'],
    'Все признаки': [
        metrics_rf_opt['test']['r2'],
        metrics_rf_opt['test']['rmse'],
        metrics_rf_opt['test']['mae']
    ],
    'Отобранные признаки': [
        metrics_rf_reduced['test']['r2'],
        metrics_rf_reduced['test']['rmse'],
        metrics_rf_reduced['test']['mae']
    ]
})

fig_comparison = go.Figure()
fig_comparison.add_trace(go.Bar(
    name='Все признаки',
    x=comparison_data['Метрика'],
    y=comparison_data['Все признаки'],
    marker_color='steelblue',
    text=[f'{val:.4f}' for val in comparison_data['Все признаки']],
    textposition='auto'
))
fig_comparison.add_trace(go.Bar(
    name='Отобранные признаки',
    x=comparison_data['Метрика'],
    y=comparison_data['Отобранные признаки'],
    marker_color='coral',
    text=[f'{val:.4f}' for val in comparison_data['Отобранные признаки']],
    textposition='auto'
))

fig_comparison.update_layout(
    title={
        'text': 'Сравнение метрик: все признаки vs отобранные признаки',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Метрика',
    yaxis_title='Значение',
    barmode='group',
    template='plotly_white',
    width=800,
    height=500,
    legend_title_text="Вариант модели"
)

fig_comparison.show()

По результатам видно, что усеченная модель показала себя хуже - гэп между трейном и тестом остался прежним (даже увеличился на процент, может быть погрешностью), при этом значения метрик упали на 4-5 процентов. Делаю вывод, что признаки, отсеченные мной содержали хоть и малую по отдельности, но важную вместе информацию.

Выглядит так, как буд-то перебором был выжат максимум у случайного леса. Можно попробовать создать новые признаки./

### 2.3 Feature Engineering

In [164]:
# ============================================================================
# Добавление синтетических химических признаков после предобработки
# ============================================================================

# 1. Функция для генерации химических признаков
def add_wine_features(df):
    """Добавляет синтетические химические признаки к данным вина"""
    data = df.copy()
    
    # Общая кислотность
    data['total_acidity'] = data['fixed acidity'] + data['volatile acidity'] + data['citric acid']
    
    # Отношение летучей кислотности к постоянной (маркер качества)
    data['vol_fixed_ratio'] = data['volatile acidity'] / (data['fixed acidity'] + 1e-5)
    
    # Процент свободного SO2 (антиокислительная защита)
    data['free_so2_ratio'] = data['free sulfur dioxide'] / (data['total sulfur dioxide'] + 1e-5)
    
    # Индекс плотности и алкоголя (тело вина)
    data['alc_density_prod'] = data['alcohol'] * data['density']
    
    # Отношение сахара к алкоголю (баланс сладости и крепости)
    data['sugar_alc_ratio'] = data['residual sugar'] / (data['alcohol'] + 1e-5)
    
    return data

# 2. Создаем новые признаки из исходных данных (до pipeline)
# Используем исходные X_train и X_test
X_train_new_features = add_wine_features(X_train)
X_test_new_features = add_wine_features(X_test)

# Извлекаем только новые признаки
new_feature_names = ['total_acidity', 'vol_fixed_ratio', 'free_so2_ratio', 
                     'alc_density_prod', 'sugar_alc_ratio']
X_train_new = X_train_new_features[new_feature_names]
X_test_new = X_test_new_features[new_feature_names]

# 3. Применяем PowerTransformer к новым признакам (как в pipeline)
# Важно: используем fit на train, transform на test
power_transformer_new = PowerTransformer(method='yeo-johnson', standardize=True)
X_train_new_transformed = power_transformer_new.fit_transform(X_train_new)
X_test_new_transformed = power_transformer_new.transform(X_test_new)

# Преобразуем в DataFrame
X_train_new_transformed = pd.DataFrame(
    X_train_new_transformed, 
    columns=new_feature_names, 
    index=X_train.index
)
X_test_new_transformed = pd.DataFrame(
    X_test_new_transformed, 
    columns=new_feature_names, 
    index=X_test.index
)

# 4. Объединяем с уже преобразованными данными
X_train_plus = pd.concat([X_train_final, X_train_new_transformed], axis=1)
X_test_plus = pd.concat([X_test_final, X_test_new_transformed], axis=1)

print("="*80)
print("Добавление синтетических признаков:")
print("="*80)
print(f"Исходные признаки: {X_train_final.shape[1]}")
print(f"Новые признаки: {len(new_feature_names)}")
print(f"Итого признаков: {X_train_plus.shape[1]}")
print(f"\nНовые признаки: {new_feature_names}")
print(f"\nРазмерность данных:")
print(f"  X_train_plus: {X_train_plus.shape}")
print(f"  X_test_plus: {X_test_plus.shape}")

Добавление синтетических признаков:
Исходные признаки: 11
Новые признаки: 5
Итого признаков: 16

Новые признаки: ['total_acidity', 'vol_fixed_ratio', 'free_so2_ratio', 'alc_density_prod', 'sugar_alc_ratio']

Размерность данных:
  X_train_plus: (1279, 16)
  X_test_plus: (320, 16)


In [165]:
# 5. Обновленная сетка параметров для GridSearchCV
# Учитываем, что признаков стало больше (11 + 5 = 16)
cv_strategy = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

param_grid_final = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [6, 7], 
    'min_samples_leaf': [4, 5, 6, 7], 
    'min_samples_split': [8, 10, 12],
    'max_features': [0.3, 0.4, 0.5, 0.6, 0.7],
    'ccp_alpha': [0.001, 0.002],
    'bootstrap': [True]
}

# 6. Запуск GridSearchCV с новыми признаками
rf_grid_final = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid_final,
    cv=cv_strategy,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True,
    verbose=2
)

print("\n" + "="*80)
print("Запуск финального поиска с новыми признаками...")
print("="*80)
rf_grid_final.fit(X_train_plus, y_train)

# 7. Результаты
best_rf = rf_grid_final.best_estimator_
y_train_pred_plus = best_rf.predict(X_train_plus)
y_test_pred_plus = best_rf.predict(X_test_plus)

r2_train = r2_score(y_train, y_train_pred_plus)
r2_test = r2_score(y_test, y_test_pred_plus)

print("\n" + "="*80)
print("РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ С НОВЫМИ ПРИЗНАКАМИ:")
print("="*80)
print(f"Лучшие параметры: {rf_grid_final.best_params_}")
print(f"Лучший R² (CV): {rf_grid_final.best_score_:.4f}")
print(f"\nR² Train: {r2_train:.4f}")
print(f"R² Test:  {r2_test:.4f}")
print(f"Gap:      {r2_train - r2_test:.4f}")

# Добавляем результат в глобальный список
add_result("RF Enhanced (New Features)", f"All ({X_train_plus.shape[1]})", best_rf, 
           X_train_plus, y_train, X_test_plus, y_test)


Запуск финального поиска с новыми признаками...
Fitting 5 folds for each of 960 candidates, totalling 4800 fits

РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ С НОВЫМИ ПРИЗНАКАМИ:
Лучшие параметры: {'bootstrap': True, 'ccp_alpha': 0.001, 'max_depth': 7, 'max_features': 0.5, 'min_samples_leaf': 4, 'min_samples_split': 8, 'n_estimators': 500}
Лучший R² (CV): 0.3720

R² Train: 0.6414
R² Test:  0.4813
Gap:      0.1601


In [166]:
# 8. Анализ важности признаков (включая новые)
feature_importance_final = pd.DataFrame({
    'feature': X_train_plus.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + "="*80)
print("Топ-10 важных признаков (включая новые):")
print("="*80)
print(feature_importance_final.head(10).to_string(index=False))

# Проверяем, попали ли новые признаки в топ
new_features_in_top = feature_importance_final[
    feature_importance_final['feature'].isin(new_feature_names)
]
print("\n" + "="*80)
print("Новые признаки и их важность:")
print("="*80)
print(new_features_in_top.to_string(index=False))


Топ-10 важных признаков (включая новые):
             feature  importance
    alc_density_prod    0.251481
           sulphates    0.160011
             alcohol    0.128464
     vol_fixed_ratio    0.084962
    volatile acidity    0.078222
total sulfur dioxide    0.045034
      free_so2_ratio    0.041525
           chlorides    0.028998
     sugar_alc_ratio    0.027199
             density    0.025220

Новые признаки и их важность:
         feature  importance
alc_density_prod    0.251481
 vol_fixed_ratio    0.084962
  free_so2_ratio    0.041525
 sugar_alc_ratio    0.027199
   total_acidity    0.021859


In [167]:
# Создание метрик для модели с новыми признаками
metrics_rf_plus = {
    'train': {
        'r2': r2_score(y_train, y_train_pred_plus),
        'rmse': np.sqrt(mean_squared_error(y_train, y_train_pred_plus)),
        'mae': mean_absolute_error(y_train, y_train_pred_plus)
    },
    'test': {
        'r2': r2_score(y_test, y_test_pred_plus),
        'rmse': np.sqrt(mean_squared_error(y_test, y_test_pred_plus)),
        'mae': mean_absolute_error(y_test, y_test_pred_plus)
    }
}

# Визуализация (графики)
plot_model_report_separate(
    metrics=metrics_rf_plus,
    feature_importance=feature_importance_final,
    model_name=f"Random Forest (Optimized, {X_train_plus.shape[1]} features with synthetic features)"
)

## 3. Оптимизация GradientBoostingRegressor

### 3.1 GridSearchCV на всех признаках, анализ важности и визуализация

In [168]:
# 1. Стратегия и сетка параметров
cv_strategy = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
# Пространство гиперпараметров для GradientBoosting
# ВАЖНО: max_depth и min_samples_leaf должны быть ограничены
param_grid_gb_enchanced = {
    'n_estimators': [800, 1000, 1200], # Больше деревьев для постепенного уточнения
    'learning_rate': [0.005],          # Но очень маленький шаг
    'max_depth': [4, 5],               # Оставляем небольшую глубину
    'min_samples_leaf': [15, 20],      # Сохраняем жесткие листы
    'subsample': [0.75, 0.8],          # Оставляем стохастичность
    'max_features': ['sqrt'],          # Это помогло, оставляем
    'validation_fraction': [0.1],      # Добавляем внутреннюю валидацию
    'n_iter_no_change': [50],          # Ранняя остановка (Early Stopping)
    'tol': [0.001]
}

# GridSearchCV на всех признаках
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid=param_grid_gb_enchanced,
    cv=cv_strategy,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

print("Запуск GridSearchCV для GradientBoosting...")
gb_grid.fit(X_train_final, y_train)

print("\n" + "="*80)
print("Лучшие параметры GradientBoosting:")
print("="*80)
print(gb_grid.best_params_)
print(f"Лучший R² (CV): {gb_grid.best_score_:.4f}")

# Финальная модель на всех признаках (золотой стандарт)
gb_optimized = gb_grid.best_estimator_
gb_optimized.fit(X_train_final, y_train)
y_pred_gb_opt_train = gb_optimized.predict(X_train_final)
y_pred_gb_opt = gb_optimized.predict(X_test_final)

# Метрики для визуализации
metrics_gb_opt = {
    'train': {
        'r2': r2_score(y_train, y_pred_gb_opt_train),
        'rmse': np.sqrt(mean_squared_error(y_train, y_pred_gb_opt_train)),
        'mae': mean_absolute_error(y_train, y_pred_gb_opt_train)
    },
    'test': {
        'r2': r2_score(y_test, y_pred_gb_opt),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_gb_opt)),
        'mae': mean_absolute_error(y_test, y_pred_gb_opt)
    }
}

print("\n" + "="*80)
print("Метрики GradientBoosting (Optimized, all features):")
print("="*80)
print(f"Train - R²: {metrics_gb_opt['train']['r2']:.4f}, RMSE: {metrics_gb_opt['train']['rmse']:.4f}, MAE: {metrics_gb_opt['train']['mae']:.4f}")
print(f"Test  - R²: {metrics_gb_opt['test']['r2']:.4f}, RMSE: {metrics_gb_opt['test']['rmse']:.4f}, MAE: {metrics_gb_opt['test']['mae']:.4f}")

Запуск GridSearchCV для GradientBoosting...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Лучшие параметры GradientBoosting:
{'learning_rate': 0.005, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 20, 'n_estimators': 1000, 'n_iter_no_change': 50, 'subsample': 0.8, 'tol': 0.001, 'validation_fraction': 0.1}
Лучший R² (CV): 0.3680

Метрики GradientBoosting (Optimized, all features):
Train - R²: 0.5917, RMSE: 0.5154, MAE: 0.3959
Test  - R²: 0.4810, RMSE: 0.5824, MAE: 0.4699


In [169]:
# Анализ важности признаков на оттюнингованной модели
feature_importance_gb = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': gb_optimized.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + "="*80)
print("Важность признаков (оттюнингованная модель):")
print("="*80)
print(feature_importance_gb.to_string(index=False))


Важность признаков (оттюнингованная модель):
             feature  importance
             alcohol    0.248685
           sulphates    0.183656
    volatile acidity    0.153702
total sulfur dioxide    0.082803
             density    0.073521
         citric acid    0.062952
           chlorides    0.049758
       fixed acidity    0.041759
                  pH    0.040340
 free sulfur dioxide    0.033349
      residual sugar    0.029476


In [170]:
# Визуализация метрик и важности признаков
plot_model_report_separate(
    metrics=metrics_gb_opt,
    feature_importance=feature_importance_gb,
    model_name="GradientBoosting (Optimized, all features)"
)

# Используем тот же порог важности
importance_threshold = 0.06  # 6%
print(f"\nПорог важности: {importance_threshold}")

# Признаки выше порога
selected_features_gb = feature_importance_gb[
    feature_importance_gb['importance'] >= importance_threshold
]['feature'].tolist()

print(f"Количество признаков выше порога: {len(selected_features_gb)}")
print(f"Отобранные признаки: {selected_features_gb}")
print(f"Отброшенные признаки: {set(X_train_final.columns) - set(selected_features_gb)}")


Порог важности: 0.06
Количество признаков выше порога: 6
Отобранные признаки: ['alcohol', 'sulphates', 'volatile acidity', 'total sulfur dioxide', 'density', 'citric acid']
Отброшенные признаки: {'pH', 'fixed acidity', 'chlorides', 'free sulfur dioxide', 'residual sugar'}


In [171]:
# Добавляем результат GB Optimized в глобальный список
add_result("GB Optimized (All)", "All (11)", gb_optimized, X_train_final, y_train, X_test_final, y_test)

### 3.2 Добавление новых признаков


Для градиентного бустинга я решил попробовать создать полиномиальные признаки на основе признаков из топ-3 - идея в том, что бустинг может получить доп информацию из взаимодействий признаков естественным образом (через последовательное разбиение), я сгенерирую эти взаимодействия с помощью умножения. При этом важно то, что делаю только на топ 3 чтобы избежать проклятия размерности.

In [172]:
# ============================================================================
# Создание полиномиальных признаков для Топ-3 признаков
# ============================================================================
from sklearn.preprocessing import PolynomialFeatures

# 1. Определяем Топ-3 признака из feature_importance_gb
top_3_features = feature_importance_gb['feature'].iloc[:3].tolist()
print("="*80)
print("Создание полиномиальных признаков (взаимодействий) для Топ-3 признаков:")
print("="*80)
print(f"Топ-3 признака: {top_3_features}")

# 2. Инициализируем трансформер (только взаимодействия, без квадратов и константы)
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

# 3. Генерируем новые признаки (на трейне и тесте раздельно!)
X_train_poly = poly.fit_transform(X_train_final[top_3_features])
X_test_poly = poly.transform(X_test_final[top_3_features])

# Получаем названия новых колонок (например, f1*f2)
poly_names = poly.get_feature_names_out(top_3_features)

# 4. Превращаем в DataFrame и объединяем с исходными данными
X_train_poly_df = pd.DataFrame(X_train_poly, columns=poly_names, index=X_train_final.index)
X_test_poly_df = pd.DataFrame(X_test_poly, columns=poly_names, index=X_test_final.index)

# Добавляем только те, которых еще нет (взаимодействия)
new_features = [name for name in poly_names if name not in top_3_features]
X_train_enhanced = pd.concat([X_train_final, X_train_poly_df[new_features]], axis=1)
X_test_enhanced = pd.concat([X_test_final, X_test_poly_df[new_features]], axis=1)

print(f"\nДобавлено новых признаков (взаимодействий): {len(new_features)}")
print(f"Новые признаки: {new_features}")
print(f"Общее количество признаков после расширения: {X_train_enhanced.shape[1]}")

Создание полиномиальных признаков (взаимодействий) для Топ-3 признаков:
Топ-3 признака: ['alcohol', 'sulphates', 'volatile acidity']

Добавлено новых признаков (взаимодействий): 3
Новые признаки: ['alcohol sulphates', 'alcohol volatile acidity', 'sulphates volatile acidity']
Общее количество признаков после расширения: 14


Гиперпараметры для сетки, как и в примерах выше, я уже отобрал сам дополнительно, чтобы добиться минимального гэп r^2 на трейне и тесте.

In [173]:
# ============================================================================
# GridSearchCV для оптимизации гиперпараметров на расширенном наборе признаков
# ============================================================================
print("\n" + "="*80)
print("GridSearchCV для GradientBoosting на расширенном наборе признаков:")
print("="*80)

# Используем ту же стратегию кросс-валидации
cv_strategy = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# Пространство гиперпараметров для расширенного поиска
# Расширенная сетка для более тщательного поиска лучших параметров
param_grid_gb_poly = {
    # Ставим большой потолок, Early Stopping сам остановит обучение
    'n_estimators': [2000], 
    
    # Очень низкий темп обучения — залог точности при использовании Poly признаков
    'learning_rate': [0.005, 0.008], 
    
    # Ограничиваем глубину, чтобы взаимодействия не превращались в шум
    'max_depth': [3], 
    
    # Увеличиваем минимальное количество объектов в листе (сильная регуляризация)
    'min_samples_leaf': [5],
    
    # Добавляем параметр min_impurity_decrease: дерево не будет делиться, 
    # если прирост качества ниже этого порога
    'min_impurity_decrease': [0.001, 0.01],
    
    # Ограничиваем количество признаков для каждого дерева (уменьшает корреляцию деревьев)
    'max_features': ['sqrt',0.2, 0.3, 0.4], 
    
    # Huber loss делает модель устойчивой к выбросам в Poly признаках
    'loss': ['huber'], 
    
    # Используем меньший subsample для внесения рандомизации (Out-of-Bag регуляризация)
    'subsample': [0.7, 0.8],
    
    # ПАРАМЕТРЫ ОСТАНОВКИ:
    'n_iter_no_change': [50],       # Остановиться, если 50 итераций нет улучшений
    'validation_fraction': [0.15],  # Увеличиваем до 15% для более стабильной проверки
    'tol': [1e-4]                   # Порог чувствительности для улучшения
}

# GridSearchCV на расширенном наборе признаков
gb_grid_enhanced = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid=param_grid_gb_poly,
    cv=cv_strategy,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

print("Запуск GridSearchCV для GradientBoosting на расширенном наборе признаков...")
gb_grid_enhanced.fit(X_train_enhanced, y_train)

print("\n" + "="*80)
print("Результаты GridSearchCV на расширенном наборе признаков:")
print("="*80)
print(f"Лучшие параметры: {gb_grid_enhanced.best_params_}")
print(f"Лучший R² (CV): {gb_grid_enhanced.best_score_:.4f}")

# ============================================================================
# Обучение финальной модели на расширенном наборе признаков
# ============================================================================
print("\n" + "="*80)
print("Обучение финальной модели GradientBoosting на расширенном наборе признаков:")
print("="*80)

gb_enhanced = gb_grid_enhanced.best_estimator_

# Предсказания на тренировочной и тестовой выборках
y_pred_gb_enhanced_train = gb_enhanced.predict(X_train_enhanced)
y_pred_gb_enhanced = gb_enhanced.predict(X_test_enhanced)


GridSearchCV для GradientBoosting на расширенном наборе признаков:
Запуск GridSearchCV для GradientBoosting на расширенном наборе признаков...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

Результаты GridSearchCV на расширенном наборе признаков:
Лучшие параметры: {'learning_rate': 0.008, 'loss': 'huber', 'max_depth': 3, 'max_features': 0.3, 'min_impurity_decrease': 0.01, 'min_samples_leaf': 5, 'n_estimators': 2000, 'n_iter_no_change': 50, 'subsample': 0.7, 'tol': 0.0001, 'validation_fraction': 0.15}
Лучший R² (CV): 0.3537

Обучение финальной модели GradientBoosting на расширенном наборе признаков:


In [174]:
# Метрики для визуализации
metrics_gb_enhanced = {
    'train': {
        'r2': r2_score(y_train, y_pred_gb_enhanced_train),
        'rmse': np.sqrt(mean_squared_error(y_train, y_pred_gb_enhanced_train)),
        'mae': mean_absolute_error(y_train, y_pred_gb_enhanced_train)
    },
    'test': {
        'r2': r2_score(y_test, y_pred_gb_enhanced),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_gb_enhanced)),
        'mae': mean_absolute_error(y_test, y_pred_gb_enhanced)
    }
}

# Важность признаков для модели с расширенными признаками
feature_importance_gb_enhanced = pd.DataFrame({
    'feature': X_train_enhanced.columns,
    'importance': gb_enhanced.feature_importances_
}).sort_values('importance', ascending=False)

print("\nТоп-10 самых важных признаков (включая взаимодействия):")
print(feature_importance_gb_enhanced.head(10).to_string(index=False))


Топ-10 самых важных признаков (включая взаимодействия):
                   feature  importance
                   alcohol    0.211361
                 sulphates    0.126935
          volatile acidity    0.106647
      total sulfur dioxide    0.075356
  alcohol volatile acidity    0.073939
         alcohol sulphates    0.066045
                   density    0.056757
sulphates volatile acidity    0.055025
               citric acid    0.046368
             fixed acidity    0.042528


In [175]:
# Визуализация метрик и важности признаков
plot_model_report_separate(
    metrics=metrics_gb_enhanced,
    feature_importance=feature_importance_gb_enhanced,
    model_name=f"GradientBoosting (Optimized, {X_train_enhanced.shape[1]} features with interactions)"
)

# Сравнение с золотым стандартом
r2_full = metrics_gb_opt['test']['r2']
r2_enhanced = metrics_gb_enhanced['test']['r2']
r2_diff = r2_enhanced - r2_full  # Теперь смотрим улучшение
r2_diff_percent = (r2_diff / r2_full) * 100 if r2_full > 0 else 0

print("\n" + "="*80)
print("GradientBoosting - Сравнение:")
print("="*80)
print(f"R² (все признаки): {r2_full:.4f}")
print(f"R² (с полиномиальными признаками): {r2_enhanced:.4f}")
print(f"Разница R²: {r2_diff:.4f} ({r2_diff_percent:+.2f}%)")

rmse_full = metrics_gb_opt['test']['rmse']
rmse_enhanced = metrics_gb_enhanced['test']['rmse']

print(f"\nRMSE (все признаки): {rmse_full:.4f}")
print(f"RMSE (с полиномиальными признаками): {rmse_enhanced:.4f}")
print(f"Улучшение RMSE: {rmse_full - rmse_enhanced:.4f}")


GradientBoosting - Сравнение:
R² (все признаки): 0.4810
R² (с полиномиальными признаками): 0.4984
Разница R²: 0.0174 (+3.61%)

RMSE (все признаки): 0.5824
RMSE (с полиномиальными признаками): 0.5725
Улучшение RMSE: 0.0098


Новые признаки - ['alcohol sulphates', 'alcohol volatile acidity', 'sulphates volatile acidity'] - занимают 5 6 и 8 место по важности, что в целом говорит о их влиянии на результат, но о влиянии более негативном, а именно рост переобучение по сравнению с оптимизированной моделью со всеми признакми. Возможно, дальнейший перебор гиперпараметров позволил бы выиграть пару процентов, но как будто того не стоит с теми результатами, что есть.

Можно увидеть по результатм, что получилось повысить точность на ~1-2 %, что обычно нормально для создания новых признаков - в моем случае тяжело выжать новую дополнительную информацию и взаимосвязи уже очень важных признаков. При этом, переобучение сильнее, чем на бустинге со всеми изначальными признаками. Из-за этого не вижу смысла использовать эту модель, мне важнее все-таки бороться с переобучением, а рост переобучения больше, роста точности на трейне.

In [176]:
# Добавляем результат GB Enhanced (с полиномиальными признаками) в глобальный список
add_result("GB Optimized (Poly)", f"Enhanced ({X_train_enhanced.shape[1]})", gb_enhanced, 
           X_train_enhanced, y_train, X_test_enhanced, y_test)

# V. Итоговое сравнение оптимизированных моделей


In [178]:
# ============================================================================
# ИТОГОВОЕ СРАВНЕНИЕ НА ОСНОВЕ MODEL_RESULTS
# ============================================================================

# Создаем DataFrame из накопленных результатов
comparison_final = pd.DataFrame(MODEL_RESULTS)

print("="*80)
print("ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (из MODEL_RESULTS):")
print("="*80)
print(comparison_final.round(4).to_string(index=False))

# Визуализация
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('R² Test Score (Higher is Better)', 'Gap |Train-Test| (Lower is Better)', 
                    'RMSE (Lower is Better)', 'MAE (Lower is Better)'),
    vertical_spacing=0.15
)

# R2 Test
fig.add_trace(go.Bar(x=comparison_final['Model'], y=comparison_final['R² Test'], 
                     marker_color='steelblue', name='R² Test'), row=1, col=1)

# Gap
fig.add_trace(go.Bar(x=comparison_final['Model'], y=comparison_final['Gap'], 
                     marker_color='orange', name='Gap'), row=1, col=2)

# RMSE
fig.add_trace(go.Bar(x=comparison_final['Model'], y=comparison_final['RMSE'], 
                     marker_color='crimson', name='RMSE'), row=2, col=1)

# MAE
fig.add_trace(go.Bar(x=comparison_final['Model'], y=comparison_final['MAE'], 
                     marker_color='green', name='MAE'), row=2, col=2)

fig.update_layout(height=900, width=1000, title_text="Сравнение эффективности моделей (MODEL_RESULTS)", showlegend=False)
fig.update_xaxes(tickangle=45)
fig.show()

ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (из MODEL_RESULTS):
                     Model      Features  R² Train  R² Test    Gap   RMSE    MAE
               RF Baseline      All (11)    0.9265   0.5404 0.3861 0.5481 0.4202
               GB Baseline      All (11)    0.6173   0.4414 0.1759 0.6042 0.4856
        RF Optimized (All)      All (11)    0.6964   0.5056 0.1908 0.5684 0.4525
    RF Optimized (Reduced)         Top 5    0.6490   0.4659 0.1831 0.5908 0.4698
RF Enhanced (New Features)      All (16)    0.6414   0.4813 0.1601 0.5822 0.4688
        GB Optimized (All)      All (11)    0.5917   0.4810 0.1107 0.5824 0.4699
       GB Optimized (Poly) Enhanced (14)    0.6397   0.4984 0.1413 0.5725 0.4565


В целом, все модели приблизительно равны - выбивается конечно Random Forest бейзлайн своим диким переобучением, но это удалось перебороть оптимизацией, хоть и ухудшая точность. По поводу оверфиттинга - боролся я с ним в основном подбором гиперпараметров, при этому не ограничиваясь только сетками - сам потом ручками тоже крутил, потому что сетки всегда целились у меня в наибольшую точность на метрике r2 (это моя ошибка, не разобрался как одновременно на несколько метрик нацелиться).

# VI. Промежуточные выводы

Основные выводы по работе нужно искать в конце второй части, после обучения бустингов не от sklearn. Тут пока могу сказать лишь, что все модели плюс минус одинаково себя показали, можно предположить наличие какого-то потолка в таком подходе решении задачи.

По поводу подбора гиперпараметров - опишу, какие из них я перебирал и почему. Если коротко, *то Random Forest я душил (чем я еще буду заниматься во второй части), чтобы модель не переобучалась, а для GradientBoosting подбирался максимально удачный темп обучения*:

1. RandomForest

    Для этой модели гиперпараметры подбирались так, чтобы снизить корреляцию между деревьями и ограничить их избыточную сложность.

    - max_depth (Максимальная глубина):

        Зачем: В бейзлайне деревья растут до полной чистоты узлов, что ведет к катастрофическому переобучению. Ограничение глубины заставляет модель находить общие закономерности в данных о вине, а не запоминать каждый конкретный образец.

    - min_samples_leaf (Мин. объектов в листе):

        Зачем: Это «сглаживающий» параметр. Он не дает модели создавать лист на основе одного-двух выбросов (редких сочетаний кислотности и сахара), что делает предсказание более устойчивым.

    - n_estimators (Количество деревьев):

        Зачем: Для леса это безопасный параметр — чем их больше, тем меньше дисперсия ошибки. Мы подбирали число, при котором точность выходит на «плато», чтобы не тратить лишние вычислительные ресурсы.

    - max_features:
    
        Зачем: Ограничение числа признаков при каждом сплите гарантирует, что деревья в лесу будут разными. Это критично для ансамбля, так как усреднение похожих деревьев не дает выигрыша в качестве.

2. Gradient Boosting (Градиентный бустинг)

    Здесь оптимизация была тоньше, так как бустинг строит деревья последовательно, исправляя ошибки предыдущих.

    - learning_rate (Скорость обучения):

        Зачем: Самый важный параметр. Он определяет «шаг», с которым мы движемся к минимуму ошибки. Слишком высокий шаг приведет к тому, что модель проскочит оптимум; слишком низкий — потребует огромного количества деревьев.

    - n_estimators (в связке с learning_rate):

        Зачем: В отличие от случайного леса, в бустинге слишком много деревьев ведут к переобучению. Мы искали баланс: если мы уменьшаем learning_rate, мы обязаны пропорционально увеличить n_estimators.

    - subsample (Подвыборка):

        Зачем: Использование не всех данных для обучения каждого следующего дерева (Stochastic Gradient Boosting). Это вносит элемент случайности и помогает модели не зацикливаться на специфических шумах в данных о качестве вина.

    - max_depth (обычно небольшая):

        Зачем: Бустинг лучше всего работает на «слабых учениках» (неглубоких деревьях). Мы оптимизировали глубину (обычно в пределах 3–6), чтобы каждое дерево исправляло конкретную ошибку смещения, а не пыталось решить всю задачу сразу.